# Consensus species timetree for miRGeneDB orthologs

Builds a species-code-labeled phylogenetic tree covering every species in `mirgenedb_species.tsv`, for use as the reference tree in downstream ortholog/consensus analysis.

**Inputs**
- `mirgenedb_species.tsv` — species codes, common names, and NCBI taxids used by the pipeline
- `timetree_species_list.nwk` — a Newick tree exported from [TimeTree](http://timetree.org) for the species names in `timetree_species_list.txt` (generated externally, between steps 2 and 3)

**Output**
- `consensus_timetree.nwk` — Newick tree with leaves labeled by pipeline species code, no internal node names/confidence values

**Pipeline**
1. Fetch NCBI taxonomy (scientific name, genus, lineage) for each taxid
2. Export the species list for submission to TimeTree
3. Check which species TimeTree's tree is missing
4. For each missing species, find its closest relative already in the tree (longest shared NCBI lineage prefix)
5. Graft missing species onto the tree next to their closest relative
6. Relabel leaves from scientific name to pipeline species code
7. Strip internal node names/confidence so the tree is clean for downstream tools

## 1. Fetch NCBI taxonomy data

Queries NCBI's `efetch` (taxonomy db) in batches of 100 for every taxid in `mirgenedb_species.tsv`, and writes scientific name, rank, genus, parent taxid, and full lineage to `mirgenedb_species_scientific_names.tsv`.


In [ ]:
"""Fetch NCBI taxonomy data for taxids in mirgenedb_species.tsv."""

import csv
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

IN_PATH = "mirgenedb_species.tsv"
OUT_PATH = "mirgenedb_species_scientific_names.tsv"

EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
BATCH_SIZE = 100  


def parse_taxon(taxon):
    scientific_name = taxon.findtext("ScientificName", "")
    rank = taxon.findtext("Rank", "")

    genus = ""
    lineage_ex = taxon.find("LineageEx")
    if lineage_ex is not None:
        for ancestor in lineage_ex.findall("Taxon"):
            if ancestor.findtext("Rank") == "genus":
                genus = ancestor.findtext("ScientificName", "")
                break

    species = ""
    if rank == "species" and genus and scientific_name.startswith(genus + " "):
        species = scientific_name[len(genus) + 1 :]

    return {
        "scientific_name": scientific_name,
        "rank": rank,
        "genus": genus,
        "species": species,
        "parent_taxid": taxon.findtext("ParentTaxId", ""),
        "lineage": taxon.findtext("Lineage", ""),
    }


def fetch_taxonomy(taxids):
    """Return {taxid: {scientific_name, rank, genus, species, parent_taxid, lineage}}."""
    info = {}
    for i in range(0, len(taxids), BATCH_SIZE):
        batch = taxids[i : i + BATCH_SIZE]
        params = {
            "db": "taxonomy",
            "id": ",".join(batch),
            "retmode": "xml",
        }
        url = f"{EFETCH_URL}?{urllib.parse.urlencode(params)}"
        with urllib.request.urlopen(url) as resp:
            root = ET.fromstring(resp.read())

        for taxon in root.findall("Taxon"):
            taxid = taxon.findtext("TaxId", "")
            info[taxid] = parse_taxon(taxon)

        time.sleep(0.4)
    return info


def main():
    with open(IN_PATH, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        rows = [row for row in reader if row.get("taxid")]

    taxids = [row["taxid"] for row in rows]
    info = fetch_taxonomy(taxids)

    missing = [t for t in taxids if t not in info]
    if missing:
        print(f"Warning: no taxonomy record found for taxids: {missing}")

    fieldnames = [
        "species_column",
        "common_name",
        "scientific_name",
        "genus",
        "species",
        "rank",
        "parent_taxid",
        "lineage",
        "taxid",
    ]
    with open(OUT_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, delimiter="\t", fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            data = info.get(row["taxid"], {})
            writer.writerow(
                {
                    "species_column": row["species_column"],
                    "common_name": row["common_name"],
                    "scientific_name": data.get("scientific_name", ""),
                    "genus": data.get("genus", ""),
                    "species": data.get("species", ""),
                    "rank": data.get("rank", ""),
                    "parent_taxid": data.get("parent_taxid", ""),
                    "lineage": data.get("lineage", ""),
                    "taxid": row["taxid"],
                }
            )

    print(f"Wrote {len(rows)} rows to {OUT_PATH}")


if __name__ == "__main__":
    main()


## 2. Export species list for TimeTree

Writes the scientific names to `timetree_species_list.txt`, one per line, for submission to TimeTree's batch tree tool. **Manual step:** submit this file to timetree.org and save the resulting Newick export as `timetree_species_list.nwk` before running the next cell.


In [ ]:
import csv

TIMETREE_OUT_PATH = "timetree_species_list.txt"

with open("mirgenedb_species_scientific_names.tsv", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    scientific_names = [row["scientific_name"] for row in reader if row.get("scientific_name")]

with open(TIMETREE_OUT_PATH, "w") as f:
    f.write("\n".join(scientific_names) + "\n")

print(f"Wrote {len(scientific_names)} species names to {TIMETREE_OUT_PATH}")


## 3. Check tree coverage

Loads the TimeTree Newick export and reports which requested species have no matching leaf. A species is considered present if its full name, or a trailing-word-trimmed version of it (e.g. a subspecies name trimmed to its species name), appears in the tree.


In [ ]:
from Bio import Phylo

TXT_PATH = "timetree_species_list.txt"
NWK_PATH = "timetree_species_list.nwk"

with open(TXT_PATH) as f:
    requested = [line.strip() for line in f if line.strip()]

tree = Phylo.read(NWK_PATH, "newick")
tree_leaves = {leaf.name.replace("_", " ") for leaf in tree.get_terminals()}


def in_tree(name):
    """True if name (or a trailing-word-trimmed version, e.g. subspecies -> species) is in the tree."""
    words = name.split()
    for n in range(len(words), 1, -1):
        if " ".join(words[:n]) in tree_leaves:
            return True
    return False


missing = [name for name in requested if not in_tree(name)]

print(f"Requested: {len(requested)}  |  In tree: {len(tree_leaves)}  |  Missing: {len(missing)}")
for name in missing:
    print(" -", name)


## 4. Find the closest relative for each missing species

For every missing species, scores every present species by how many leading taxonomic ranks its NCBI lineage shares with the missing species' lineage (longest common prefix), and reports the best-matching present species (or tied set).


In [ ]:
import csv

TSV_PATH = "mirgenedb_species_scientific_names.tsv"

with open(TSV_PATH, newline="") as f:
    rows = [r for r in csv.DictReader(f, delimiter="\t") if r.get("scientific_name")]
by_name = {r["scientific_name"]: r for r in rows}


present = [r for r in rows if r["scientific_name"] not in set(missing)]
missing_rows = [by_name[n] for n in missing if n in by_name]


def lineage_list(row):
    return [x.strip() for x in row["lineage"].split(";") if x.strip()]


def lcp_len(a, b):
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


for m in missing_rows:
    m_lin = lineage_list(m)
    scored = sorted(
        ((lcp_len(m_lin, lineage_list(p)), p["scientific_name"]) for p in present),
        reverse=True,
    )
    best_score = scored[0][0]
    best = [name for score, name in scored if score == best_score]
    shared = m_lin[best_score - 1] if best_score else "(none)"
    print(f"{m['scientific_name']:35s} -> {', '.join(best):40s} (shared: {shared}, depth {best_score})")


## 5. Graft missing species onto the tree

Attaches each missing species as a new leaf, reusing the branch length of its closest relative. A single best match is grafted as that relative's sibling; a tie is grafted at the common ancestor of all tied candidates, since lineage data alone can't break the tie. Writes `timetree_species_list_augmented.nwk`.


In [ ]:
from Bio.Phylo.BaseTree import Clade

AUGMENTED_NWK_PATH = "timetree_species_list_augmented.nwk"


def get_parent(tree, child):
    for clade in tree.find_clades():
        if child in clade.clades:
            return clade
    return None


name_to_leaf = {leaf.name.replace("_", " "): leaf for leaf in tree.get_terminals()}

graft_log = []
for m in missing_rows:
    m_lin = lineage_list(m)
    scored = sorted(
        ((lcp_len(m_lin, lineage_list(p)), p["scientific_name"]) for p in present),
        reverse=True,
    )
    best_score = scored[0][0]
    best_names = [name for score, name in scored if score == best_score]
    candidate_clades = [name_to_leaf[n] for n in best_names]

    # Single closest match: graft as its sibling, same branch length.
    # Multiple tied matches: graft at their common ancestor (the shared taxonomic rank),
    # since lineage data alone can't break the tie among equally-related present species.
    if len(candidate_clades) == 1:
        attach_point = get_parent(tree, candidate_clades[0])
        branch_length = candidate_clades[0].branch_length
    else:
        attach_point = tree.common_ancestor(candidate_clades)
        branch_length = tree.distance(attach_point, candidate_clades[0])

    new_name = m["scientific_name"].replace(" ", "_")
    attach_point.clades.append(Clade(branch_length=branch_length, name=new_name))
    graft_log.append((m["scientific_name"], best_names[:3], len(best_names), round(branch_length, 2)))

Phylo.write(tree, AUGMENTED_NWK_PATH, "newick")

print(f"Grafted {len(graft_log)} species. New leaf count: {tree.count_terminals()}")
print(f"Wrote {AUGMENTED_NWK_PATH}")
for name, cands, n_tied, bl in graft_log:
    tail = f" (+{n_tied - 1} tied)" if n_tied > 1 else ""
    print(f"  {name:30s} attached near {cands[0]}{tail}, branch_length={bl}")


## 6. Relabel leaves with pipeline species codes

Maps each tree leaf (scientific name) back to the short species code used elsewhere in the pipeline, trimming trailing words (e.g. subspecies) when no exact match exists. Writes `consensus_timetree.nwk`.


In [ ]:
import csv

from Bio import Phylo

SPECIES_TSV = "mirgenedb_species_scientific_names.tsv"
AUGMENTED_NWK_PATH = "timetree_species_list_augmented.nwk"
CODE_NWK_PATH = "consensus_timetree.nwk"

with open(SPECIES_TSV, newline="") as f:
    code_to_name = {
        r["species_column"]: r["scientific_name"]
        for r in csv.DictReader(f, delimiter="\t")
        if r.get("species_column")
    }

tree = Phylo.read(AUGMENTED_NWK_PATH, "newick")
tree_leaves = {leaf.name.replace("_", " ") for leaf in tree.get_terminals()}


name_to_code = {}
unmatched = []
for code, name in code_to_name.items():
    if name in tree_leaves:
        name_to_code[name] = code
        continue
    words = name.split()
    matched = False
    for n in range(len(words) - 1, 1, -1):
        prefix = " ".join(words[:n])
        if prefix in tree_leaves:
            name_to_code[prefix] = code
            matched = True
            break
    if not matched:
        unmatched.append(code)

if unmatched:
    print(f"WARNING: {len(unmatched)} species codes have no match in the tree: {unmatched}")

renamed = 0
for leaf in tree.get_terminals():
    key = leaf.name.replace("_", " ")
    if key in name_to_code:
        leaf.name = name_to_code[key]
        renamed += 1
    else:
        print(f"WARNING: no species code found for tree leaf '{leaf.name}'")

Phylo.write(tree, CODE_NWK_PATH, "newick")
print(f"Renamed {renamed}/{tree.count_terminals()} leaves. Wrote {CODE_NWK_PATH}")


## 7. Strip internal node metadata

Removes internal node names and confidence values so the final tree is a clean, minimal Newick file for downstream tools.


In [ ]:
from Bio import Phylo

CODE_NWK_PATH = "consensus_timetree.nwk"

tree = Phylo.read(CODE_NWK_PATH, "newick")
for clade in tree.find_clades():
    if not clade.is_terminal():
        clade.name = None
        clade.confidence = None

Phylo.write(tree, CODE_NWK_PATH, "newick")
print(f"Removed internal node names from {CODE_NWK_PATH}")
